In [2]:
import pandas as pd
import numpy as np

# Load the datasets into Pandas DataFrames
users = pd.read_csv('datasets/Users.csv', encoding='latin-1', on_bad_lines='skip', low_memory=False)
ratings = pd.read_csv('datasets/Ratings.csv', encoding='latin-1', on_bad_lines='skip', low_memory=False)
books = pd.read_csv('datasets/Books.csv', encoding='latin-1', on_bad_lines='skip', low_memory=False)

# Print out the first 5 rows of each file so we can inspect them
print("--- USERS DATA ---")
print(users.head())

print("\n--- RATINGS DATA ---")
print(ratings.head())

print("\n--- BOOKS DATA ---")
print(books.head())

--- USERS DATA ---
  User-ID  Age
0       1  NaN
1       2   18
2       3  NaN
3       4   17
4       5  NaN

--- RATINGS DATA ---
   User-ID       ISBN  Rating
0        2  195153448     0.0
1        7   34542252     0.0
2        8    2005018     5.0
3        8   60973129     0.0
4        8  374157065     0.0

--- BOOKS DATA ---
        ISBN                                              Title  \
0  195153448                                Classical Mythology   
1    2005018                                       Clara Callan   
2   60973129                               Decision in Normandy   
3  374157065  Flu: The Story of the Great Influenza Pandemic...   
4  393045218                             The Mummies of Urumchi   

                 Author    Year                Publisher Unnamed: 5  \
0    Mark P. O. Morford  2002.0  Oxford University Press        NaN   
1  Richard Bruce Wright  2001.0    HarperFlamingo Canada        NaN   
2          Carlo D'Este  1991.0          HarperPerenn

In [3]:
users['Age'] = pd.to_numeric(users['Age'], errors='coerce')

# Define "invalid" ages. Let's say anyone under 5 or over 100 is invalid.
# We temporarily turn those invalid ages into blanks (NaN) too.
users.loc[(users['Age'] < 5) | (users['Age'] > 100), 'Age'] = np.nan

# Calculate the mean (average) of the remaining valid ages.
mean_age = users['Age'].mean()
print(f"The calculated mean of valid ages is: {mean_age:.2f}")

# Fill ALL the blanks (NaN) with this mean age!
users['Age'] = users['Age'].fillna(mean_age)

The calculated mean of valid ages is: 34.72


In [4]:
# Prove that it worked by checking if there are any blanks left!
print("\n--- AFTER CLEANING AGE ---")
print(f"Total blank 'Age' values left: {users['Age'].isnull().sum()}")
print(users.head(10))


--- AFTER CLEANING AGE ---
Total blank 'Age' values left: 0
  User-ID        Age
0       1  34.721849
1       2  18.000000
2       3  34.721849
3       4  17.000000
4       5  34.721849
5       6  61.000000
6       7  34.721849
7       8  34.721849
8       9  34.721849
9      10  26.000000


In [5]:
# Finally, let's make sure the 'Age' column is now a nice clean integer type (no decimals).
users['Age'] = users['Age'].round(0).astype(int)
print("\n--- FINAL CHECK ---")
print(users.head(10))


--- FINAL CHECK ---
  User-ID  Age
0       1   35
1       2   18
2       3   35
3       4   17
4       5   35
5       6   61
6       7   35
7       8   35
8       9   35
9      10   26


In [6]:
# Count missing values in Title, Author, and Publisher
missing_text = books[['Title', 'Author', 'Publisher']].isnull().sum()

print("--- BLANK VALUES IN BOOKS DATA ---")
print(missing_text)

# Find the total number of rows that have AT LEAST one of these missing
total_rows_with_blanks = books[['Title', 'Author', 'Publisher']].isnull().any(axis=1).sum()
print(f"\nTotal rows that will be dropped: {total_rows_with_blanks}")

--- BLANK VALUES IN BOOKS DATA ---
Title            0
Author       37211
Publisher    38539
dtype: int64

Total rows that will be dropped: 38541


In [7]:
# Count total rows before doing anything
total_rows_before = len(books)
print(f"Total rows before: {total_rows_before}")

# Find how many have blanks in Title, Author, or Publisher
total_rows_with_blanks = books[['Title', 'Author', 'Publisher']].isnull().any(axis=1).sum()
print(f"Total rows to be dropped: {total_rows_with_blanks}")

# ACTUALLY drop those rows 
books = books.dropna(subset=['Title', 'Author', 'Publisher'])

# Count total rows left after dropping
total_rows_after = len(books)
print(f"Total rows after: {total_rows_after}")

Total rows before: 268031
Total rows to be dropped: 38541
Total rows after: 229490


In [8]:
# Force the Year column to be numbers (turn weird text into blanks/NaN)
books['Year'] = pd.to_numeric(books['Year'], errors='coerce')

# Find the absolute smallest and largest years in the raw dataset
smallest_year = books['Year'].min()
largest_year = books['Year'].max()
year_range = largest_year - smallest_year

In [9]:
print("--- RAW YEAR COLUMN EXPLORATION ---")
print(f"Smallest Year found: {smallest_year}")
print(f"Largest Year found: {largest_year}")
print(f"Total Range of Years: {year_range} years\n")

--- RAW YEAR COLUMN EXPLORATION ---
Smallest Year found: 0.0
Largest Year found: 2050.0
Total Range of Years: 2050.0 years



In [10]:
# Define "Valid Years" (between 1800 and 2026) to calculate accurate stats
valid_years = books.loc[(books['Year'] >= 1800) & (books['Year'] <= 2026), 'Year']

# Calculate Mean and Median of the valid years
mean_year = valid_years.mean()
median_year = valid_years.median()

print("--- VALID YEAR ANALYSIS (1800 - 2026) ---")
print(f"Mean (Average) Year: {mean_year:.0f}")
print(f"Median (Middle) Year: {median_year:.0f}")

--- VALID YEAR ANALYSIS (1800 - 2026) ---
Mean (Average) Year: 1994
Median (Middle) Year: 1996


In [11]:
# Turn any impossible years (before 1800 or after 2026) into blanks (NaN)
books.loc[(books['Year'] < 1800) | (books['Year'] > 2026), 'Year'] = np.nan

# Fill ALL the blanks with the mean year we calculated earlier
books['Year'] = books['Year'].fillna(mean_year)

# Round the years and convert them to clean integers (whole numbers)
books['Year'] = books['Year'].round(0).astype(int)

In [12]:
# Verify that it worked perfectly!
print("--- AFTER CLEANING YEAR ---")
print(f"Total blank or invalid 'Year' values left: {books['Year'].isnull().sum()}")

--- AFTER CLEANING YEAR ---
Total blank or invalid 'Year' values left: 0


In [13]:
# Create a "Translation Dictionary" for Users (starts at 1)
user_translation = {old_id: new_id for new_id, old_id in enumerate(users['User-ID'].unique(), start=1)}

# Create a "Translation Dictionary" for Books (starts at 1)
book_translation = {old_id: new_id for new_id, old_id in enumerate(books['ISBN'].unique(), start=1)}

In [14]:
# Apply the translation to the Users file
users['UserID'] = users['User-ID'].map(user_translation)

# Apply the translation to the Books file
books['BookID'] = books['ISBN'].map(book_translation)

In [15]:
# Apply BOTH translations to the Ratings file so it matches!
ratings['UserID'] = ratings['User-ID'].map(user_translation)
ratings['BookID'] = ratings['ISBN'].map(book_translation)

# Verify our work! Let's peek at the Ratings file to see the new columns.
print("--- UPDATED RATINGS DATA ---")
print(ratings[['User-ID', 'UserID', 'ISBN', 'BookID', 'Rating']].head())

--- UPDATED RATINGS DATA ---
   User-ID  UserID       ISBN  BookID  Rating
0        2     NaN  195153448     1.0     0.0
1        7     NaN   34542252     NaN     0.0
2        8     NaN    2005018     2.0     5.0
3        8     NaN   60973129     3.0     0.0
4        8     NaN  374157065     4.0     0.0


In [16]:
# Finding ghost-books that are in the Ratings file but not in the Books file

# Count the total number of ratings we started with
total_ratings = len(ratings)

# Count how many ratings belong to "Ghost Books" (BookID is NaN)
ghost_books_count = ratings['BookID'].isnull().sum()

# Count how many ratings belong to "Ghost Users" (UserID is NaN)
ghost_users_count = ratings['UserID'].isnull().sum()

# print("--- GHOST DATA INSPECTION ---")
print(f"Total ratings originally: {total_ratings}")
print(f"Ratings attached to unknown Books: {ghost_books_count}")
print(f"Ratings attached to unknown Users: {ghost_users_count}")

Total ratings originally: 1048575
Ratings attached to unknown Books: 220318
Ratings attached to unknown Users: 1048575


In [17]:
# Force original IDs to perfectly match as strings to fix formatting mismatches
users['User-ID'] = users['User-ID'].astype(str).str.strip()
ratings['User-ID'] = ratings['User-ID'].astype(str).str.strip()
books['ISBN'] = books['ISBN'].astype(str).str.strip()
ratings['ISBN'] = ratings['ISBN'].astype(str).str.strip()

# Gather EVERY unique User-ID by combining the Users and Ratings files
all_user_ids = pd.concat([users['User-ID'], ratings['User-ID']]).unique()

# Gather EVERY unique ISBN by combining the Books and Ratings files
all_isbns = pd.concat([books['ISBN'], ratings['ISBN']]).unique()

In [18]:
# Create the Translation Dictionaries from these massive, combined lists!
user_translation = {old_id: new_id for new_id, old_id in enumerate(all_user_ids, start=1)}
book_translation = {old_id: new_id for new_id, old_id in enumerate(all_isbns, start=1)}

# Apply the translations to the Users and Books files
users['UserID'] = users['User-ID'].map(user_translation)
books['BookID'] = books['ISBN'].map(book_translation)

# Apply the translations to the Ratings file
ratings['UserID'] = ratings['User-ID'].map(user_translation)
ratings['BookID'] = ratings['ISBN'].map(book_translation)

# Let's prove it worked! Count the NaNs again.
ghost_books_left = ratings['BookID'].isnull().sum()
ghost_users_left = ratings['UserID'].isnull().sum()

In [19]:
print("--- NEW GHOST DATA INSPECTION ---")
print(f"Total NaN BookIDs left: {ghost_books_left}")
print(f"Total NaN UserIDs left: {ghost_users_left}")

--- NEW GHOST DATA INSPECTION ---
Total NaN BookIDs left: 0
Total NaN UserIDs left: 0


In [20]:
# Force ratings to be numbers
ratings['Rating'] = pd.to_numeric(ratings['Rating'], errors='coerce')

# Identify valid ratings (between 0 and 10)
valid_ratings = ratings.loc[(ratings['Rating'] >= 0) & (ratings['Rating'] <= 10), 'Rating']

# Count how many ratings are invalid (below 0, above 10, or blank)
invalid_ratings_count = len(ratings) - len(valid_ratings)
print(f"Total invalid ratings (below 0 or above 10): {invalid_ratings_count}")

# Calculate the mean of valid ratings and round it to a whole number
mean_rating = valid_ratings.mean()
rounded_mean_rating = int(round(mean_rating))
print(f"Mean of valid ratings: {mean_rating:.2f} (Rounding to {rounded_mean_rating})")

Total invalid ratings (below 0 or above 10): 5
Mean of valid ratings: 2.88 (Rounding to 3)


In [21]:
# Replace all invalid ratings and blanks with your rounded mean
ratings.loc[(ratings['Rating'] < 0) | (ratings['Rating'] > 10) | (ratings['Rating'].isnull()), 'Rating'] = rounded_mean_rating

# Verify the fix!
invalid_left = len(ratings.loc[(ratings['Rating'] < 0) | (ratings['Rating'] > 10)])
print(f"Invalid ratings left: {invalid_left}")

Invalid ratings left: 0


In [22]:
# Start with the ratings table, and bring in the 'Age'
merged_step1 = pd.merge(ratings[['UserID', 'BookID', 'Rating']], users[['UserID', 'Age']], on='UserID', how='left')

# Bring in the book details
final_df = pd.merge(merged_step1, books[['BookID', 'Title', 'Author', 'Year', 'Publisher']], on='BookID', how='left')

# Rearrange the columns to match your EXACT requested order
final_df = final_df[['UserID', 'Age', 'BookID', 'Title', 'Author', 'Year', 'Publisher', 'Rating']]

# Rename the columns exactly how you typed them
final_df.columns = ['UserID', 'age', 'BookID', 'title', 'author', 'year', 'publisher', 'rating']

# Sort the entire dataset by UserID so it counts 1, 2, 3... perfectly!
final_df = final_df.sort_values(by=['UserID', 'BookID']).reset_index(drop=True)

In [23]:
# Save this masterpiece
final_df.to_csv('datasets/Master_Dataset.csv', index=False)

# Print a preview
print("--- FINAL SORTED MASTER DATASET ---")
print(final_df.head(10))

--- FINAL SORTED MASTER DATASET ---
   UserID  age  BookID                                              title  \
0       2   18       1                                Classical Mythology   
1       7   35  229490                                                NaN   
2       8   35       2                                       Clara Callan   
3       8   35       3                               Decision in Normandy   
4       8   35       4  Flu: The Story of the Great Influenza Pandemic...   
5       8   35       5                             The Mummies of Urumchi   
6       8   35       6                             The Kitchen God's Wife   
7       8   35       7  What If?: The World's Foremost Military Histor...   
8       8   35       8                                    PLEADING GUILTY   
9       8   35       9  Under the Black Flag: The Romance and the Real...   

                 author    year                 publisher  rating  
0    Mark P. O. Morford  2002.0   Oxford Univers

In [24]:
from scipy.sparse import csr_matrix

# Add 1 because Python counts from 0, but our IDs start at 1.
num_users = final_df['UserID'].max() + 1
num_books = final_df['BookID'].max() + 1

# Build the Sparse Matrix!
sparse_matrix = csr_matrix(
    (final_df['rating'], (final_df['UserID'], final_df['BookID'])), 
    shape=(num_users, num_books)
)

In [25]:
print("--- SPARSE MATRIX SUCCESSFULLY BUILT ---")
print(f"Matrix shape (Users, Books): {sparse_matrix.shape}")
print(f"Total actual ratings stored inside: {sparse_matrix.nnz}")

--- SPARSE MATRIX SUCCESSFULLY BUILT ---
Matrix shape (Users, Books): (278856, 333609)
Total actual ratings stored inside: 1048306


In [26]:
from sklearn.datasets import dump_svmlight_file

# LIBSVM requires a "target" (y) to predict. 
dummy_labels = np.zeros(sparse_matrix.shape[0])

# Export the matrix and the dummy labels to the LIBSVM format!
dump_svmlight_file(sparse_matrix, dummy_labels, 'datasets/Recommender_Matrix.libsvm')

print("--- EXPORT COMPLETE! ---")
print("Your sparse matrix is successfully saved as 'Recommender_Matrix.libsvm' in your datasets folder.")

--- EXPORT COMPLETE! ---
Your sparse matrix is successfully saved as 'Recommender_Matrix.libsvm' in your datasets folder.


In [27]:
# Slice out a tiny piece of the giant matrix (first 100 users, first 100 books)
tiny_slice = sparse_matrix[0:100, 0:100]

# Convert this small sparse slice into a normal, dense grid (filling in the zeros)
dense_slice = tiny_slice.todense()

# Turn it into a Pandas DataFrame so we can save it
slice_df = pd.DataFrame(dense_slice, columns=[f"Book_{i}" for i in range(100)])

# Save this tiny preview to a CSV
slice_df.to_csv('datasets/Matrix_Preview_100x100.csv', index=False)

print("--- PREVIEW CSV GENERATED ---")
print("Saved 'Matrix_Preview_100x100.csv' to your datasets folder.")
print("\nHere is a peek at the top-left corner of your grid (Notice all the zeros!):")
print(slice_df.head(10))

--- PREVIEW CSV GENERATED ---
Saved 'Matrix_Preview_100x100.csv' to your datasets folder.

Here is a peek at the top-left corner of your grid (Notice all the zeros!):
   Book_0  Book_1  Book_2  Book_3  Book_4  Book_5  Book_6  Book_7  Book_8  \
0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
1     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
2     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
3     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
4     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
5     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
6     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
7     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
8     0.0     0.0     5.0     0.0     0.0     0.0     0.0     0.0     0.0   
9     0.0     0.0     0.0     0.0     0.0     0.0     0.0     0